# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR<sup>2</sup> dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
* [Croissant Schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed in the current environment
!pip install mlcroissant

## 1. Data Loading
Let's load the dataset metadata and exploration handle using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset: ", metadata.name)
print("Description: ", metadata.description)
print(f"License: {metadata.license}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview

Now let's inspect the available record sets, and enumerate fields and sample records. For all references, we use the `@id`.

> **Note:** The `record_sets`, `fields`, and columns are all referenced by their `@id` for clarity and reproducibility.

In [ ]:
# List available record set IDs in the dataset
record_set_ids = [x["@id"] for x in getattr(metadata, 'record_sets', getattr(metadata, 'recordSet', []))]
if not record_set_ids:
    # Try to infer record sets from distribution->hasPart and from mlcroissant's convenience method
    record_set_ids = [r['@id'] for r in dataset.record_sets]

print(f"Record Sets (@id): {record_set_ids}")

# List fields for each record set
for rs_id in record_set_ids:
    print(f"\nRecord set: {rs_id}")
    record_set = dataset.get_record_set(rs_id)
    field_ids = [field['@id'] for field in (record_set.get('field', []) if record_set.get('field') else [])]
    print(f"  Fields (@id): {field_ids}")
    if field_ids:
        print("  Sample fields:")
        for field_id in field_ids[:5]:
            field = dataset.get_field(field_id)
            print(f"    - {field_id}: {field.get('name', '')}")

# If no explicit fields, print a sample record
if record_set_ids:
    print("\nSample record(s) from first record set:")
    first_rs = record_set_ids[0]
    for i, rec in enumerate(dataset.records(record_set=first_rs)):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction

Let's extract data from the main record set into a pandas DataFrame for analysis. 

> All operations reference the record set, field, and column using their `@id` variables for reproducibility.

In [ ]:
# We'll use the first available record set as 'main'.
record_sets = record_set_ids
dataframes = {}

# Load records from each record set (if there are several, else just main)
for rs_id in record_sets:
    print(f"Loading records from record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f'  Columns: {dataframes[rs_id].columns.tolist()}')
            print(dataframes[rs_id].head())
        else:
            print('  No records found.')
    except Exception as e:
        print(f'  Error loading records: {e}')

# For later steps, select main data
main_rs_id = record_sets[0] if record_sets else None
# Show the first few column names
if main_rs_id:
    print("Columns in main record set:", dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate filtering, normalization, and grouping with one or two numeric fields and one group/categorical field.

- Use field/column `@id`s for selection.
- Show basic statistics and handle missing data if necessary.

In [ ]:
# Identify suitable numeric fields and group field by inspecting columns
df = dataframes[main_rs_id]
print("All available columns:", df.columns.tolist())

# Guess likely numeric field candidates:
numeric_candidates = [col for col in df.columns if any(key in col.lower() for key in ["mw", "molecular_weight", "count", "quantity", "abundance", "coverage", "number", "peptide"])]
if not numeric_candidates:
    # Fallback: just try the first float column
    numeric_candidates = df.select_dtypes(['number']).columns.tolist()
print('Likely numeric fields:', numeric_candidates)

# Guess grouping field
group_field_candidates = [col for col in df.columns if any(term in col.lower() for term in ["sample", "group", "type", "condition"])]
print('Possible group fields:', group_field_candidates)

# Select a numeric field and group field
numeric_field_id = numeric_candidates[0] if numeric_candidates else None
group_field_id = group_field_candidates[0] if group_field_candidates else None

print(f"Using numeric field: {numeric_field_id}")
print(f"Using grouping field: {group_field_id}")

if numeric_field_id:
    # Convert to numeric, if needed
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75)

    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} above 75th percentile ({threshold}):")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count', 'min', 'max'])
        )
        print(f"\nGrouped data by {group_field_id} (mean/count/min/max of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the main numeric field, and group-wise statistics if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

if group_field_id and numeric_field_id:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR<sup>2</sup> dataset on human protein abundance in extracellular vesicles using the `mlcroissant` library. We:

- Loaded the full metadata from the Croissant schema.
- Inspected available record sets, fields, and previewed a few records using their `@id`s.
- Loaded a main record set to a DataFrame and identified key numeric and categorical fields.
- Filtered and normalized numeric data, and performed group-wise aggregations.
- Plotted distributions and group differences.

This template can be extended for further downstream analysis, e.g., more advanced visualizations, outlier analysis, annotation integration, and machine learning workflows, all while referencing each entity by its Croissant `@id`.
